# 03 · Dois algoritmos disputam o modelo do ciclo

**Etapa do fluxo:** 6 (treinamento do Modelo A e do Modelo B).

- **Modelo A:** `LogisticRegression` (linear, interpretável, referência forte em dados tabulares).
- **Modelo B:** `HistGradientBoostingClassifier` (árvores com *boosting*, captura interações não lineares).

Protocolo idêntico: **mesmas features eleitas** no notebook 02, mesmo treino, mesma validação, mesma seed e mesma
métrica principal: **ROC-AUC**. A taxa de conversão varia de ~5% a ~49% entre os lotes; o F1 depende do limiar e
dessa prevalência, enquanto a AUC mede a capacidade de **ordenar** clientes por chance de conversão - que é o uso real
(priorizar quem ligar primeiro). F1 macro, acurácia balanceada, precisão e recall são registrados como secundárias.
Os dois usam `class_weight="balanced"`. O conjunto de teste não é usado aqui.

In [1]:
import sys
import warnings
from pathlib import Path

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config" / "config.yaml").exists())
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

import json
import matplotlib.pyplot as plt
import pandas as pd

from src.config import caminho, carregar_config

pd.set_option("display.max_columns", 50)
cfg = carregar_config(RAIZ)

In [2]:
import mlflow
from mlflow.models import infer_signature
from src.ciclo import atualizar_contexto, carregar_contexto, diretorio_ciclo, obter_cycle_id
from src.modelagem import avaliar_baseline, colunas_entrada, treinar_candidato
from src.rastreamento import run_etapa

cycle_id = obter_cycle_id(cfg)
contexto = carregar_contexto(cfg, cycle_id)
pasta_ciclo = diretorio_ciclo(cfg, cycle_id)
processado = caminho(cfg, cfg["dados"]["diretorio_processado"])
treino = pd.read_csv(processado / "treino.csv")
validacao = pd.read_csv(processado / "validacao.csv")
eleitas = contexto["features_eleitas"]
metrica = cfg["avaliacao"]["metrica_principal"]
print(cycle_id, contexto["data_version"], "| features:", eleitas)

c20261123T185659Z-gh45243588298 v005-6563af51 | features: ['euribor3m', 'contatado_antes', 'emp.var.rate', 'cons.price.idx', 'poutcome', 'contact', 'nr.employed', 'month', 'cons.conf.idx']


In [3]:
baseline = avaliar_baseline(treino, validacao, cfg)
print("Baseline (classe majoritária) na validação:", baseline)

candidatos = {}
for chave, especificacao in cfg["modelos"].items():
    resultado = treinar_candidato(especificacao, eleitas, treino, validacao, cfg)
    with run_etapa(cfg, contexto, "treino_candidato", nome=f"{chave}-{especificacao['nome']}", tags={"candidato": chave, "algoritmo": especificacao["nome"]}) as run:
        mlflow.log_params({
            "algoritmo": especificacao["nome"], "classe": especificacao["classe"], "seed": cfg["projeto"]["seed"],
            "features": ",".join(eleitas), "n_features": len(eleitas), "metrica_principal": metrica,
            **{f"hp_{k}": v for k, v in especificacao.get("parametros", {}).items()},
        })
        mlflow.log_metrics({f"val_{k}": v for k, v in resultado["metricas_validacao"].items() if isinstance(v, float)})
        mlflow.log_metrics({f"treino_{k}": v for k, v in resultado["metricas_treino"].items() if isinstance(v, float)})
        mlflow.log_metric("tempo_treino_s", resultado["tempo_treino_s"])
        mlflow.log_dict({"features": eleitas}, "features_eleitas.json")
        amostra = validacao[colunas_entrada(cfg)].head(50)
        mlflow.sklearn.log_model(resultado["pipeline"], "modelo", signature=infer_signature(amostra, resultado["pipeline"].predict(amostra)))
        run_id = run.info.run_id
    candidatos[chave] = {"nome": especificacao["nome"], "run_id": run_id, "metricas_validacao": resultado["metricas_validacao"], "metricas_treino": resultado["metricas_treino"]}
    print(f"{chave} ({especificacao['nome']}): {metrica} validação = {resultado['metricas_validacao'][metrica]:.4f} | run_id {run_id}")

Baseline (classe majoritária) na validação: {'f1_macro': 0.4714, 'balanced_accuracy': 0.5, 'accuracy': 0.8917, 'precision_pos': 0.0, 'recall_pos': 0.0, 'n': 7444, 'taxa_positiva_real': 0.1083, 'roc_auc': 0.5}


modelo_a (logistic_regression): roc_auc validação = 0.7990 | run_id 9a991687171146b0a36ff6d5962cf497


modelo_b (hist_gradient_boosting): roc_auc validação = 0.8064 | run_id 07bbc079709d4c02a58c488cfbc7fdfe


## Comparação lado a lado

In [4]:
comparacao = pd.DataFrame({
    f"{c} · {d['nome']}": {**{f"val_{k}": v for k, v in d["metricas_validacao"].items()}, **{f"treino_{k}": v for k, v in d["metricas_treino"].items()}}
    for c, d in candidatos.items()
})
comparacao["baseline"] = pd.Series({f"val_{k}": v for k, v in baseline.items()})
comparacao.to_csv(pasta_ciclo / "comparacao_modelos.csv")
display(comparacao)

linhas = ["val_f1_macro", "val_roc_auc", "val_balanced_accuracy", "val_recall_pos", "val_precision_pos"]
comparacao.loc[linhas].plot.bar(figsize=(10, 4), title=f"Validação - ciclo {cycle_id}", rot=20)
plt.ylim(0, 1)
plt.show()

contexto = atualizar_contexto(cfg, cycle_id, candidatos=candidatos, baseline_validacao=baseline)

,modelo_a · logistic_regression,modelo_b · hist_gradient_boosting,baseline
val_f1_macro,0.6794,0.6705,0.4714
val_balanced_accuracy,0.7425,0.7429,0.5000
val_accuracy,0.8383,0.8272,0.8917
val_precision_pos,0.3577,0.3404,0.0000
val_recall_pos,0.6203,0.6352,0.0000
val_n,7444.0000,7444.0000,7444.0000
val_taxa_positiva_real,0.1083,0.1083,0.1083
val_roc_auc,0.7990,0.8064,0.5000
treino_f1_macro,0.6792,0.6816,NaN
treino_balanced_accuracy,0.7390,0.7526,NaN
